# 02 - LST pipeline (Phase 2)

**Colombo UHI practicum.** Builds the temperature layer every later phase reads:

1. a **harmonised Landsat 5/7/8/9 Collection-2 Level-2 collection** - QA_PIXEL
   bits 0-4 + `QA_RADSAT == 0` masking, per-sensor scale factors, and surface
   reflectance renamed to `blue`...`swir2` so index code is sensor-agnostic;
2. **annual and dry-season (Jan-Mar) composites**, each shipping a **per-pixel
   valid-observation count**;
3. **MODIS MOD11A2 / MYD11A2** day and night LST with explicit `QC_Day` /
   `QC_Night` bit filtering (good quality AND average error <= 1 K);
4. a **Landsat-vs-MODIS annual mean comparison** over the CMC.

Run top-to-bottom in **Google Colab** after `01_aoi_and_boundaries.ipynb` has
worked once. All logic lives in `src/colombo_uhi/`; this notebook orchestrates
and displays.

> **Caveat (CLAUDE.md #1):** every number here is **LAND SURFACE TEMPERATURE**,
> never air temperature. Surface UHI can be roughly 2x the canopy-air UHI.
>
> **Caveat (CLAUDE.md #2):** never read a composite without its `obs_count`
> band. Tropical cloud cover means only a minority of scenes are usable.
>
> **Caveat (CLAUDE.md #4):** Landsat sees one ~10:30 local overpass. Night-time
> UHI comes only from MODIS.

### If Earth Engine says "User memory limit exceeded"

It is almost always **graph depth**, not pixel count. The rule of thumb: work
out what gets embedded into each image of a long collection, and how many times.
Levers in order of effect - all are already applied below, so this list is for
when you extend the notebook:

1. **Do not embed a composited mask in a long series.** `aoi.water_mask`
   internally composites Landsat; masking 26 annual images with it instantiates
   that composite 26 times. Use `aoi.static_water_mask` (a single JRC image) for
   series work. *This one changes the numbers slightly - measure it.*
2. **Ask for fewer bands** (`include_sr=False`, `include_st_qa=False`). Scaling
   six reflectance bands on ~1670 scenes is real weight when you only want LST.
3. **Drop the percentile band** (`with_percentile=False`). A percentile reducer
   retains every observation per pixel to sort them; mean and count stream.
4. **Select years at construction time, never by filtering.** A collection built
   from `ee.List.map()` materialises *every* element's graph when you `.filter()`
   it, so `filter(year == 2000)` costs the same as all 26 years. Build the years
   you want with `start_year`/`end_year` instead — that is what
   `zonal_annual_means_by_year` and Step 4 do.
5. **Shrink `WORK_REGION`** (Step 1) and raise `ZONAL_SCALE_M` (Step 6).

Only levers 1 and 5 move a number. The rest change how the work is packaged.

In [ ]:
# COLAB: RUN THIS CELL
# Clone the repo on first run; fast-forward pull on later runs.
import os

REPO_URL = "https://github.com/Dineth0627/colombo_uhi.git"
REPO_DIR = "/content/" + REPO_URL.rstrip("/").removesuffix(".git").rsplit("/", 1)[-1]

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git pull --ff-only

# Which revision is actually on disk. Quote this if a result looks impossible.
!git --no-pager log -1 --format="HEAD %h %s (%ci)"

In [ ]:
# COLAB: RUN THIS CELL  (skip if you already ran notebook 00 or 01 in this runtime)
%pip install -q -r requirements.txt
print("\nIf Colab asked to RESTART the runtime: Runtime > Restart session,")
print("then re-run this notebook FROM THE CLONE CELL (skip this pip cell).")

In [ ]:
# COLAB: RUN THIS CELL
# Load params (single source of truth) and initialise Earth Engine.
import sys

sys.path.insert(0, os.path.abspath("src"))

# Drop any already-imported colombo_uhi modules BEFORE importing. Without this,
# re-running the notebook in a live runtime keeps the version cached in
# sys.modules from the previous run: `git pull` updates the files on disk but the
# import silently returns the OLD code, so new functions appear to not exist
# (AttributeError) and fixed bugs appear unfixed. This cost a full run in Phase 1d.
for _name in [m for m in list(sys.modules) if m == "colombo_uhi" or m.startswith("colombo_uhi.")]:
    del sys.modules[_name]

from colombo_uhi import load_params
from colombo_uhi.auth import init_ee

params = load_params()
project = init_ee()
print("Earth Engine initialised with project:", project)
print()
for _key in ("lst_not_air_temp", "valid_obs_required", "single_overpass"):
    print("CAVEAT:", " ".join(params["caveats"][_key].split()))
    print()

## Step 1 - geometries and the water mask, scoped tight

Everything downstream is built over `WORK_REGION` = **Colombo District**, not
the province-plus-25 km `analysis_region` that Phase 1 used for the rural ring.
The reflectance composite behind `aoi.water_mask` is built over whatever region
it is given, so this one choice dominates the memory cost of the whole notebook.

**Water is masked before any statistic** (CLAUDE.md). That matters more than it
sounds for Colombo: COD-AB's CMC polygon encloses ~6.9 km2 of Port outer
harbour, and leaving it in would drag the urban mean down.

In [ ]:
# COLAB: RUN THIS CELL
import ee

from colombo_uhi import aoi, composites, indices, landsat, modis, viz

# Fail fast and legibly if a STALE colombo_uhi is loaded (see the purge above).
_required = {
    "landsat": ["harmonised_collection", "monsoon_season", "output_band_names"],
    "composites": ["annual_composites", "dry_season_composites", "scene_inventory"],
    "modis": ["lst_collection", "annual_lst", "clear_sky_count"],
    "indices": ["add_indices", "albedo"],
    "viz": ["plot_annual_lst_comparison"],
}
_absent = {
    name: [f for f in funcs if not hasattr(globals()[name], f)]
    for name, funcs in _required.items()
}
_absent = {k: v for k, v in _absent.items() if v}
if _absent:
    raise RuntimeError(
        f"colombo_uhi modules are STALE, missing {_absent}.\n"
        f"  landsat loaded from: {landsat.__file__}\n"
        "Fix, in order:\n"
        "  1. Runtime > Restart session, then re-run from the CLONE cell.\n"
        "  2. If it persists, your local commits are not pushed - check that the\n"
        "     HEAD line printed by the clone cell is the revision you expect."
    )

district_fc = aoi.colombo_district(params)
district_geom = district_fc.geometry(10)

# Degrade gracefully if the CMC cannot be built, so the rest still runs
# (same pattern as notebook 01).
try:
    cmc_geom = aoi.cmc_boundary(params)
    print("CMC boundary built.")
except Exception as exc:  # noqa: BLE001 - surface the real reason, keep going
    cmc_geom = None
    print("COULD NOT BUILD THE CMC:", exc)
    print("Falling back to Colombo District for the zonal statistics below.")

zone_geom = cmc_geom if cmc_geom is not None else district_geom
zone_label = "CMC" if cmc_geom is not None else "Colombo District"

# THE memory knob for this notebook. Shrink it first if EE runs out of memory.
WORK_REGION = district_geom.bounds(10)

print("Zonal statistics will use:", zone_label)
print("Working region area (km2):", aoi.area_km2(WORK_REGION).getInfo())

In [ ]:
# COLAB: RUN THIS CELL
# Build the water mask ONCE, over the working region only. It composites
# Landsat internally, so both the scope and the reuse matter.
water = aoi.water_mask(params, region=WORK_REGION)
land = water.Not()
print("Water mask built over the working region.")
print("Thresholds:", {k: v for k, v in params["aoi"]["water_mask"].items() if k != "composite"})

## Step 2 - the harmonised Landsat collection

Four collections merged into one, every scene carrying `LST_C` (degrees Celsius)
plus `blue`/`green`/`red`/`nir`/`swir1`/`swir2` and `ST_QA_K`.

The diagnostic prints scene counts **with and without** the `PROCESSING_LEVEL`
filter, which drops `L2SR` scenes (surface reflectance only, thermal fully
masked). It is implemented as `neq("L2SR")` rather than `eq("L2SP")` precisely
so a renamed property cannot silently empty the collection - but check the drop
is small anyway.

In [ ]:
# COLAB: RUN THIS CELL
scenes = landsat.harmonised_collection(params, region=WORK_REGION)

# Band names come from the client-side helper, NOT from bandNames().getInfo():
# asking Earth Engine for them would force it to evaluate the whole image graph.
print("Harmonised band schema:", landsat.output_band_names(params))
print("Total scenes over the working region:", scenes.size().getInfo())
print()

_c2l2 = params["landsat_c2l2"]
_start = f"{params['time']['start_year']}-01-01"
_end = f"{params['time']['end_year'] + 1}-01-01"
print(f"{'sensor':<12}{'unfiltered':>12}{'L2SP only':>12}")
for _key in landsat.sensor_keys(params):
    _raw = (
        ee.ImageCollection(params["datasets"][_key]["id"])
        .filterBounds(WORK_REGION)
        .filterDate(_start, _end)
    )
    _kept = _raw.filter(
        ee.Filter.neq(_c2l2["processing_level_property"], _c2l2["processing_level_exclude"])
    )
    print(f"{_key:<12}{_raw.size().getInfo():>12}{_kept.size().getInfo():>12}")
print()
print("If an 'L2SP only' count is 0 while 'unfiltered' is not, the property name")
print("has changed: set landsat_c2l2.processing_level_filter_enabled: false.")

## Step 3 - scenes per year x sensor: where the data gaps are

Expect these gaps, and treat anything else as suspicious:

| Period | Expectation |
|---|---|
| 2000-2003 | Landsat 5 + Landsat 7, both SLC-on - the best early coverage |
| from 2003-06 | Landsat 7 striped (SLC-off); kept by default, median compositing dilutes it |
| 2012-05 to 2013-03 | **Landsat 7 only** - L5 ended 2012-05-05, L8 launched 2013-03-18 |
| from 2013-03 | Landsat 8 |
| from 2021-10 | Landsat 9 joins; L7 collection ends 2024-01 |

These are metadata-only reductions - one Earth Engine round trip each, no pixels.

In [ ]:
# COLAB: RUN THIS CELL
dry_months = params["time"]["seasons"]["dry_window"]["months"]
dry_scenes = landsat.harmonised_collection(params, region=WORK_REGION, months=dry_months)

inventory_all = composites.scene_inventory(scenes, params)
inventory_dry = composites.scene_inventory(dry_scenes, params)

print("SCENES PER YEAR x SENSOR - full year")
print(inventory_all.to_string())
print()
print(f"SCENES PER YEAR x SENSOR - dry window (months {dry_months})")
print(inventory_dry.to_string())
print()
_empty = inventory_dry.index[inventory_dry["total"] == 0].tolist()
print("Dry-window years with ZERO scenes:", _empty if _empty else "none")

## Step 4 - dry-season composite for a recent year

`dry_season_composites` restricts to Jan-Mar (the driest, clearest window) and
returns one image per year with three bands: `LST_C` (median), `LST_C_p90`, and
`obs_count`.

Only the mapped year is built here (`start_year == end_year`). Building all 26
and filtering down to one would force Earth Engine to evaluate every year's
graph just to read the `year` property off each - that is what exhausted the
memory limit on the first run.

In [ ]:
# COLAB: RUN THIS CELL
MAP_YEAR = 2025  # most recent complete Jan-Mar window inside the study period

recent = ee.Image(
    composites.dry_season_composites(
        scenes, params, start_year=MAP_YEAR, end_year=MAP_YEAR
    ).first()
).updateMask(land)

# Band names client-side; only metadata is fetched, in ONE round trip.
_expected_bands = composites.composite_band_names(
    params["landsat_c2l2"]["lst_band_name"], params["composites"]["percentile"], params
)
_meta = ee.Dictionary(
    {
        "n_scenes": recent.get(params["composites"]["n_scenes_property"]),
        "reducer": recent.get("reducer"),
        "percentile": recent.get("percentile"),
        "year": recent.get(params["composites"]["year_property"]),
    }
).getInfo()

print(f"Dry-season composite {MAP_YEAR}")
print("  bands       :", _expected_bands)
print("  metadata    :", _meta)

In [ ]:
# COLAB: RUN THIS CELL
# Static PNGs into figures/ - the interactive map renders nothing once the
# notebook is saved, so these are the reviewable evidence.
from IPython.display import Image, display

# Display-only stretches (not analysis constants).
LST_MIN_C, LST_MAX_C = 22, 42
OBS_MAX = 12
THERMAL = ["2c7bb6", "abd9e9", "ffffbf", "fdae61", "d7191c"]
COUNTS = ["440154", "31688e", "35b779", "fde725"]

outlines = [viz.outline_image(district_fc, "000000", 2)]
if cmc_geom is not None:
    outlines.append(viz.outline_image(cmc_geom, "9467bd", 3))

lst_png = viz.save_thumbnail(
    [recent.select("LST_C").visualize(min=LST_MIN_C, max=LST_MAX_C, palette=THERMAL)]
    + outlines,
    WORK_REGION,
    f"figures/lst_dry_season_{MAP_YEAR}.png",
)
print(f"LST (degC), dry season {MAP_YEAR} - stretch {LST_MIN_C}-{LST_MAX_C} degC")
display(Image(filename=str(lst_png)))

In [ ]:
# COLAB: RUN THIS CELL
obs_png = viz.save_thumbnail(
    [recent.select(params["composites"]["obs_count_band"])
        .visualize(min=0, max=OBS_MAX, palette=COUNTS)]
    + outlines,
    WORK_REGION,
    f"figures/lst_obs_count_{MAP_YEAR}.png",
)
print(f"VALID OBSERVATIONS behind that composite - stretch 0-{OBS_MAX} scenes")
print("Read this together with the map above. CLAUDE.md caveat 2: a warm pixel")
print("backed by one scene is not evidence of anything.")
display(Image(filename=str(obs_png)))

In [ ]:
# COLAB: RUN THIS CELL
# Observation-count statistics over the analysis zone - the honest floor for
# Phase 4 trend fitting.
obs_band = params["composites"]["obs_count_band"]
obs_stats = (
    recent.select(obs_band)
    .reduceRegion(
        reducer=ee.Reducer.min()
        .combine(ee.Reducer.median(), sharedInputs=True)
        .combine(ee.Reducer.max(), sharedInputs=True),
        geometry=zone_geom,
        scale=params["crs"]["analysis_scale_m"],
        maxPixels=params["composites"]["reduce_max_pixels"],
        tileScale=params["composites"]["tile_scale"],
    )
    .getInfo()
)
print(f"Dry-season {MAP_YEAR} valid observations over {zone_label}: {obs_stats}")
print()
print("min == 0 means some pixels had NO usable scene that dry season. That is a")
print("real tropical-cloud result, not a bug - Phase 4 must mask on it, not hide it.")

## Step 5 - MODIS day and night LST

MOD11A2 is a plain average of the daily retrievals with **no built-in quality
filtering**, so `QC_Day` / `QC_Night` are applied explicitly here: mandatory QA
"good quality" (bits 0-1 == 0) **and** average LST error <= 1 K (bits 6-7 == 0).

MODIS is the only night-time source in this project (CLAUDE.md caveat 4). Terra
overpasses ~10:30 / ~22:30 local, Aqua ~13:30 / ~01:30 - Aqua daytime is closest
to peak heating, Terra daytime closest to the Landsat overpass.

Aqua carries no data before **2002-07-04**; `clamp_start_date` warns and clamps
rather than returning empty years that look like data gaps. At 1 km these
collections are cheap compared with Landsat.

**Day and night use different QC thresholds, and that is deliberate.** In run 5
the strict day policy (mandatory QA == 0 AND error <= 1 K) returned **zero**
night pixels over the CMC for all 26 years on both satellites - not a
conservative answer, no answer at all. `modis_lst.qc_filter.night` is therefore
relaxed to mandatory QA <= 1 and error <= 3 K. The cell below prints the QC
class distribution so you can see for yourself which field was doing the
killing, rather than taking that on trust.

**Carry into Phase 3:** night LST is accepted at up to 3 K stated uncertainty
against <= 1 K for day, so night-time SUHII is weaker evidence than daytime
SUHII and must never be reported as an equal-confidence pair.

In [ ]:
# COLAB: RUN THIS CELL
# WHY was night empty? Tally the raw QC classes - no LST masking - and compare
# against the configured ceilings. Mass sitting above the ceiling is the cause.
qc_cfg = params["modis_lst"]["qc_filter"]
print("configured ceilings (<=):")
for _overpass in ("day", "night"):
    print(f"  {_overpass:<6} mandatory_qa <= {qc_cfg[_overpass]['mandatory_qa_max']}"
          f", lst_error <= {qc_cfg[_overpass]['lst_error_max']}")
print()

for _daynight in ("day", "night"):
    hist = modis.qc_class_histogram("terra", _daynight, params, zone_geom)
    print(f"--- MOD11A2 QC_{_daynight.title()} over the {zone_label} ---")
    for _field in ("mandatory_qa", "lst_error"):
        ceiling = qc_cfg[_daynight][f"{_field}_max"]
        rows = hist[hist["field"] == _field]
        kept = rows[rows["class"] <= ceiling]["share"].sum()
        print(f"  {_field} (keeping class <= {ceiling}: {kept:.1%} of pixels)")
        for _, row in rows.iterrows():
            flag = "KEEP" if row["class"] <= ceiling else "drop"
            print(f"    [{flag}] class {int(row['class'])}: {row['share']:6.1%}  {row['label']}")
    print()

In [ ]:
# COLAB: RUN THIS CELL
modis_scale = params["modis_lst"]["reduction_scale_m"]
modis_series = {}

for product in ("terra", "aqua"):
    for daynight in ("day", "night"):
        # Left UNMASKED here; Step 6 applies the same static land mask to every
        # plotted series so they are all masked identically.
        modis_series[(product, daynight)] = modis.annual_lst(
            product, daynight, params, geometry=zone_geom, region=WORK_REGION
        )

print("MODIS annual composites built:", sorted(modis_series))
print("Overpass times:", params["modis_lst"]["overpass_local_time"])
print()

# One QC sanity check: how much does the filtering actually remove?
_terra_day = modis.lst_collection("terra", "day", params, region=WORK_REGION)
print("Terra day 8-day granules over the study period:", _terra_day.size().getInfo())

## Step 6 - Landsat vs MODIS annual means over the CMC

The two are **not expected to agree in absolute terms**. What matters is whether
they agree in **shape**; the offset is a finding to report, not an error to tune
away. Sources of the offset:

* **30 m vs 1 km.** A 1 km MODIS pixel over central Colombo mixes roofs, roads
  and canopy; the Landsat mean over the same polygon weights them differently.
* **Sampling.** Landsat is one instantaneous ~10:30 overpass per 16 days, on
  clear days only. MODIS is an 8-day average of clear-sky daily retrievals.
* **Overpass time.** Terra day (~10:30) is the fair comparator to Landsat;
  Aqua day (~13:30) sits nearer peak heating and should read warmer.

Five deliberate choices below. Four are about fitting inside the Earth Engine
memory limit; only the last one moves a number, and it is measured rather than
assumed.

* **One reducer for both sides** (`mean`). The project default composites
  Landsat with a median and MODIS with a mean; leaving that mismatch in would
  put a reducer artefact inside a number everyone reads as a sensor difference.
* **`include_sr=False`.** The comparison needs `LST_C` only, so scaling and
  renaming six reflectance bands on each of ~1670 scenes is pure graph weight.
* **`with_percentile=False`.** A percentile reducer must retain every
  observation per pixel in order to sort them; mean and count are streaming
  accumulators.
* **Year batching at CONSTRUCTION time** (`zonal_annual_means_by_year`). This is
  the subtle one. An annual series is built with
  `ee.ImageCollection.fromImages(ee.List.map(...))`, so calling
  `.filter(year == 2000)` on it makes Earth Engine materialise **all 26**
  composite graphs just to test the predicate — batching by filter saves
  nothing, which is why one-year-per-request still failed. The builder instead
  calls `annual_composites(start_year=lo, end_year=hi)` per batch, so only those
  years exist. Changes no numbers.
* **A static (JRC-only) water mask for the long series.** This is the one that
  moves a number. `aoi.water_mask` ORs three detectors, two of which composite a
  Landsat collection - and that composite gets embedded into every one of the 26
  annual images. `aoi.static_water_mask` is a single image from JRC Global
  Surface Water occurrence. For permanent water (ocean, Port harbour, Beira,
  Kelani) the two agree closely; the static one misses seasonal and shallow
  water. **The next cell measures the difference on 2025 instead of asserting it
  is negligible.**

The Landsat and MODIS series are fetched in separate cells, so a failure in one
does not cost you the other.

In [ ]:
# COLAB: RUN THIS CELL
# How much does the cheap static water mask actually change a CMC mean? Measured
# on the 2025 dry-season composite, which is already built, so this is two small
# reductions - not an assumption.
static_water = aoi.static_water_mask(params, region=WORK_REGION)
static_land = static_water.Not()

_reducer = ee.Reducer.mean()
_kw = dict(
    geometry=zone_geom,
    scale=params["crs"]["analysis_scale_m"],
    maxPixels=params["composites"]["reduce_max_pixels"],
    tileScale=params["composites"]["tile_scale"],
)
_full = recent.select("LST_C").reduceRegion(reducer=_reducer, **_kw).getInfo()["LST_C"]
_static = (
    ee.Image(composites.dry_season_composites(
        scenes, params, start_year=MAP_YEAR, end_year=MAP_YEAR
    ).first())
    .updateMask(static_land)
    .select("LST_C")
    .reduceRegion(reducer=_reducer, **_kw)
    .getInfo()["LST_C"]
)

print(f"CMC mean dry-season LST {MAP_YEAR}, by water mask:")
print(f"  combined (MNDWI OR QA OR JRC) : {_full:.4f} degC")
print(f"  static (JRC occurrence only)  : {_static:.4f} degC")
print(f"  difference                    : {_static - _full:+.4f} degC")
print()
print("Report this number with the comparison plot. If it is a few hundredths of")
print("a degree the static mask is a fair substitute for the long series; if it")
print("is large, the series below inherits that bias and must say so.")

In [ ]:
# COLAB: RUN THIS CELL
# This cell makes ~13 small requests (26 years / BATCH_YEARS) and takes a few
# minutes. Progress prints per batch so you can see it advancing.
COMPARISON_REDUCER = "mean"
ZONAL_SCALE_M = 100   # see the note above
BATCH_YEARS = 2       # lower to 1 if Earth Engine still reports a memory limit

# include_sr/include_st_qa False -> LST-only scenes, no reflectance scaling
lst_only_scenes = landsat.harmonised_collection(
    params, region=WORK_REGION, include_sr=False, include_st_qa=False
)

series = {}
landsat_label = f"Landsat 5/7/8/9 ({COMPARISON_REDUCER}, {ZONAL_SCALE_M} m, ~10:30)"
series[landsat_label] = composites.zonal_annual_means_by_year(
    lst_only_scenes,
    zone_geom,
    params,
    reducer=COMPARISON_REDUCER,
    mask=static_land,
    scale_m=ZONAL_SCALE_M,
    batch_years=BATCH_YEARS,
    progress=True,
)
print(f"\n--- {landsat_label} ---")
print(series[landsat_label].to_string(index=False))

In [ ]:
# COLAB: RUN THIS CELL
# MODIS at 1 km is far cheaper, but it goes through the same by-year builder so
# every plotted series is composited and masked identically.
for product in ("terra", "aqua"):
    for daynight in ("day", "night"):
        label = f"MODIS {product.title()} {daynight} (1 km)"
        granules = modis.lst_collection(product, daynight, params, region=WORK_REGION)
        series[label] = composites.zonal_annual_means_by_year(
            granules,
            zone_geom,
            params,
            reducer=COMPARISON_REDUCER,
            mask=static_land,
            scale_m=modis_scale,
            batch_years=BATCH_YEARS,
        )
        print(f"--- {label} ---")
        print(series[label].to_string(index=False))
        print()

In [ ]:
# COLAB: RUN THIS CELL
comparison_png = viz.plot_annual_lst_comparison(
    series,
    "figures/lst_landsat_vs_modis_cmc.png",
    params,
    title=f"Annual mean LST over the {zone_label}, {params['time']['start_year']}-"
          f"{params['time']['end_year']}",
)
display(Image(filename=str(comparison_png)))

print("Reminder: these are SURFACE temperatures under clear skies only. A")
print("clear-sky sampling bias warms every series here relative to a true annual")
print("mean, and it is not corrected anywhere in this pipeline.")
print()
print("The lower panel is not decoration: some MODIS points rest on a handful of")
print("1 km pixels while every Landsat point rests on ~4200. Read them together.")

### Decomposing the Landsat-vs-MODIS offset

Run 5 showed Landsat at ~39-41 degC against MODIS Terra day at ~32-35 degC - at
the *same* overpass time, and warmer than Aqua day (13:30, nearer peak heating),
which is the wrong way round. Three causes are tangled together:

| Cause | Test |
|---|---|
| **Too few pixels** - the MODIS CMC mean rests on 13-23 1 km pixels, some years as few as 2 | rerun over Colombo District (~700 MODIS pixels) |
| **Resolution / mixing** - 30 m resolves hot roofs; a 1 km pixel averages in parks, water and suburb | reduce Landsat at 1000 m over the same polygon |
| **Genuine sensor / emissivity difference** | whatever offset survives both tests |

The next three cells run those tests and print the residual. Nothing here
"corrects" anything - the point is to be able to say which part of the gap is
which when the report quotes it.

In [ ]:
# COLAB: RUN THIS CELL
# TEST 1 - aggregation-matched: the same Landsat composites, reduced at 1 km.
landsat_1km_label = f"Landsat ({COMPARISON_REDUCER}, 1000 m, ~10:30)"
series[landsat_1km_label] = composites.zonal_annual_means_by_year(
    lst_only_scenes,
    zone_geom,
    params,
    reducer=COMPARISON_REDUCER,
    mask=static_land,
    scale_m=modis_scale,
    batch_years=BATCH_YEARS,
)
print(f"--- {landsat_1km_label} ---")
print(series[landsat_1km_label].to_string(index=False))

In [ ]:
# COLAB: RUN THIS CELL
# TEST 2 - district scope, where MODIS has a real sample instead of ~20 pixels.
district_series = {}
district_series[f"Landsat ({COMPARISON_REDUCER}, {ZONAL_SCALE_M} m)"] = (
    composites.zonal_annual_means_by_year(
        lst_only_scenes,
        district_geom,
        params,
        reducer=COMPARISON_REDUCER,
        mask=static_land,
        scale_m=ZONAL_SCALE_M,
        batch_years=BATCH_YEARS,
        progress=True,
    )
)
for _daynight in ("day", "night"):
    label = f"MODIS Terra {_daynight} (1 km)"
    district_series[label] = composites.zonal_annual_means_by_year(
        modis.lst_collection("terra", _daynight, params, region=WORK_REGION),
        district_geom,
        params,
        reducer=COMPARISON_REDUCER,
        mask=static_land,
        scale_m=modis_scale,
        batch_years=BATCH_YEARS,
    )

for label, frame in district_series.items():
    print(f"--- {label} (Colombo District) ---")
    print(frame.to_string(index=False))
    print()

district_png = viz.plot_annual_lst_comparison(
    district_series,
    "figures/lst_landsat_vs_modis_district.png",
    params,
    title="Annual mean LST over Colombo District, "
          f"{params['time']['start_year']}-{params['time']['end_year']}",
)
display(Image(filename=str(district_png)))

In [ ]:
# COLAB: RUN THIS CELL
# The decomposition. Each row is the mean Landsat-minus-Terra-day difference
# over the years both series cover.
import pandas as pd

def mean_offset(a, b):
    merged = pd.merge(
        a[["year", "mean"]], b[["year", "mean"]], on="year", suffixes=("_a", "_b")
    ).dropna()
    return merged["mean_a"].sub(merged["mean_b"]).mean(), len(merged)

terra_day_cmc = series["MODIS Terra day (1 km)"]
comparisons = [
    (f"{zone_label}, Landsat 100 m  vs Terra day", series[landsat_label], terra_day_cmc),
    (f"{zone_label}, Landsat 1000 m vs Terra day", series[landsat_1km_label], terra_day_cmc),
    ("District, Landsat 100 m  vs Terra day",
     district_series[f"Landsat ({COMPARISON_REDUCER}, {ZONAL_SCALE_M} m)"],
     district_series["MODIS Terra day (1 km)"]),
]
print(f"{'comparison':<46}{'mean offset':>13}{'years':>7}")
for name, left, right in comparisons:
    offset, n = mean_offset(left, right)
    print(f"{name:<46}{offset:>+12.2f}C{n:>7}")

print()
print("How to read it:")
print("  row 1 -> row 2 shrinking  = the gap was largely RESOLUTION (30 m vs 1 km)")
print("  row 1 -> row 3 shrinking  = the CMC gap was largely SMALL SAMPLE")
print("  whatever survives both    = genuine sensor/emissivity difference, and")
print("                              that is the number the report should quote")

## Step 7 - spectral indices spot check

Indices are sensor-agnostic because they read the harmonised `blue`...`swir2`
band names. Albedo uses **one coefficient set across all four sensors** - see
`config/params.yaml`, `indices.albedo`, for why that preserves rather than
breaks continuity at the 2013 sensor change.

In [ ]:
# COLAB: RUN THIS CELL
# Indices on the dry-season scenes of the mapped year only.
_year_scenes = landsat.harmonised_collection(
    params,
    region=zone_geom,
    start_date=f"{MAP_YEAR}-01-01",
    end_date=f"{MAP_YEAR}-04-01",
)
_index_composite = (
    _year_scenes.map(lambda img: indices.add_indices(img, params))
    .median()
    .updateMask(land)
)

_index_bands = list(indices.INDEX_BAND_NAMES.values())
_stats = _index_composite.select(_index_bands).reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=zone_geom,
    scale=ZONAL_SCALE_M,
    maxPixels=params["composites"]["reduce_max_pixels"],
    tileScale=params["composites"]["tile_scale"],
).getInfo()

print(f"Mean index values over the {zone_label}, Jan-Mar {MAP_YEAR} (at {ZONAL_SCALE_M} m):")
for _name in _index_bands:
    _value = _stats.get(_name)
    print(f"  {_name:<8}{_value if _value is None else round(_value, 4)}")
print()
print("Plausibility: NDVI ~0.1-0.4 over a dense city core, NDBI positive,")
print("MNDWI negative on land, albedo ~0.10-0.20 for urban fabric.")

## What to check before signing Phase 2 off

1. **Scene inventory** - 2012-05 to 2013-03 is Landsat 7 only; no Landsat 8
   before 2013; no Landsat 9 before late 2021. A year that is unexpectedly zero
   is a bug, not weather.
2. **`L2SP only` counts** are close to the unfiltered counts. A zero there means
   the `PROCESSING_LEVEL` property changed.
3. **Dry-season LST map** - the CMC core reads warmer than its vegetated
   surroundings, water is absent (masked), values roughly 25-45 degC.
4. **Observation-count map** - non-zero across the CMC. Large 0-2 areas are the
   real tropical-cloud floor; record the number, do not hide it.
5. **QC histogram** explains the night gap - night mass sitting in classes above
   the strict ceiling. If it does not, the night policy is the wrong fix and I
   need to know from this run.
6. **MODIS night series now return values**, roughly 24-28 degC, clearly cooler
   than every daytime series. No `UserWarning` about an empty series.
7. **Landsat vs MODIS** - the series track each other in shape, and the
   decomposition table attributes the offset to sample size, resolution, or a
   genuine sensor difference. Aqua day should sit warmest of the day series.
8. **Index means** are physically plausible (see the cell above).

### About the ~40 degC annual means

These are **clear-sky, ~10:30, LAND SURFACE** temperatures over a dense urban
core. They are not air temperature and are not comparable to a weather report -
surface UHI can be roughly 2x the canopy-air UHI (CLAUDE.md caveat 1). Two
further reasons they read high, neither of them corrected anywhere in this
pipeline, and both of which belong in the report:

* **clear-sky sampling bias** - Landsat only sees cloud-free moments, which are
  the hotter ones. A true annual mean surface temperature would be cooler.
* **30 m resolution over a built core** - the composite resolves hot roofs and
  asphalt that a coarser sensor averages away against parks and water.

Report back with the printed tables and the three PNGs in `figures/`, and I will
close Phase 2 or fix what they expose.

> Nothing in this notebook has been executed by Claude Code - it has no Earth
> Engine credentials. Until you run it, every Earth Engine cell here is
> unverified.